# 演習 02 — ML 分類 (scikit-learn)

Iris データセットを使った分類演習です。  
依存: `pip install scikit-learn matplotlib seaborn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
iris = load_iris()
X, y = iris.data, iris.target

print(f"データ形状: {X.shape}")
print(f"クラス: {iris.target_names}")
print(f"特徴量: {iris.feature_names}")

## 1. データ探索 (EDA)

In [ ]:
import pandas as pd

df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = [iris.target_names[i] for i in y]

print(df.groupby("species").describe().T)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
colors = ["steelblue", "coral", "seagreen"]

for ax, feature in zip(axes.flat, iris.feature_names):
    for cls, color in zip(iris.target_names, colors):
        vals = df[df["species"] == cls][feature]
        ax.hist(vals, bins=15, alpha=0.6, label=cls, color=color)
    ax.set_title(feature)
    ax.legend()

plt.suptitle("各特徴量の分布（クラス別）", fontsize=14)
plt.tight_layout()
plt.show()

## 2. モデルの訓練と評価

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True)),
    ]),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
    results[name] = {
        "test_acc": accuracy_score(y_test, y_pred),
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
    }
    print(f"{name:25s}: テスト={results[name]['test_acc']:.4f}, "
          f"CV={results[name]['cv_mean']:.4f}±{results[name]['cv_std']:.4f}")

In [ ]:
# 混同行列の可視化（ランダムフォレスト）
rf = models["Random Forest"]
cm = confusion_matrix(y_test, rf.predict(X_test))

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel("予測")
plt.ylabel("正解")
plt.title("混同行列 — ランダムフォレスト")
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf.predict(X_test),
                             target_names=iris.target_names))

## 3. グリッドサーチでハイパーパラメータを最適化する

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
gs.fit(X_train, y_train)

print("最良パラメータ:", gs.best_params_)
print(f"CV 最良スコア: {gs.best_score_:.4f}")
print(f"テストスコア:  {gs.score(X_test, y_test):.4f}")

## 4. 特徴量重要度の可視化

In [ ]:
best_model = gs.best_estimator_
importances = best_model.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar(range(len(iris.feature_names)), importances[idx], color="steelblue")
plt.xticks(range(len(iris.feature_names)),
           [iris.feature_names[i] for i in idx], rotation=15, ha="right")
plt.ylabel("重要度")
plt.title("特徴量重要度 — 最適化後ランダムフォレスト")
plt.tight_layout()
plt.show()

print("\n💡 petal 系の特徴量が重要である理由を `notes.md` で確認してください。")